In [1]:
# Install / upgrade dependencies (ConfigSpace >=0.6.0 needed for 'q' arg used by Ray BOHB)
%pip install -U tables ray ipywidgets tensorboard tensorboardX optuna vmdpy seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.9/78.9 MB 83.0 MB/s  0:00:006m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 41.8 MB/s  0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 4.0.15
    Uninstalling widgetsnbextension-4.0.15:
      Successfully uninstalled widgetsnbextension-4.0.15
  Attempting uninstall: jupyterlab_widgets
    Found existing installation: jupyterlab_widgets 3.0.16
    Uninstalling jupyterlab_widgets-3.0.16:
      Successfully uninstalled jupyterlab_widgets-3.0.16
  Attempting uninstall: optuna
    Found existing installation: optuna 4.9.0
    Uninstalling optuna-4.9.0:
      Successfully uninstalled optuna-4.9.0
  Attempting uninstall: ipywidgets━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [optuna]
    Found existing installation: ipywidgets 8.1.8━━━━━━━━━━━━━ 2/5 [optuna]
    Uninstalling ipywidgets-8.1.8:━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [optuna]
      Successfully uninstalled ipywidgets-8.1.8━━━━━━━━━━━━━━

### Imports

### Manual Transformer Training

In [ ]:
import importlib
import transformer_model
importlib.reload(transformer_model)

<module 'transformer_model' from '/home/pedro/Linux-AntigravityProjects/SurroGrid-linux/GridForecast/3_transformer/transformer_model.py'>

In [ ]:
from transformer_model import TransformerTrainer, TransformerConfig
import torch, math, os, json

manual_cfg = {
    # Preprocessing settings
    '_data': {
        'hdf_data_path': '/home/pedro/Linux-AntigravityProjects/SurroGrid-linux/GridForecast/0_preprocessing/Data/ts_train.h5',
        'key_X': "X",
        'key_y': "y",
        'train_grids': "all",
        'test_ratio': 0.2,
        'random_state': 42,
        'X_BASE_COLS':        [#'DHI', 'DNI', 'GHI', 
                               'T', 
                               'demand_net_active_pre',
                               #"mob_avail", "mobility", #"mob_last_avail", 
                               "heat_water", "heat_space", "cop_avg",
                               # "PV_prod_100", 
                               "PV_prod_expected",
                               #"PV_prod_true",
                               # Base footprint areas and total floor area
                               "res_bldng_area_base_sum", "nonres_bldng_area_base_sum", "mixed_bldng_area_base_sum", "bldng_area_floors_sum",
                               # Building counts by type (pure residential, pure non-residential, mixed)
                               'n_cars', "n_pure_res_buildings", "n_pure_nonres_buildings", "n_mixed_buildings", 'n_flats', 'n_occ', # "n_radiator", #'n_bldng',
                               'n_lines', 'tot_R_grid', 'regiostar7',
                              ],
        'TARGET_COLS':        [#'demand_net_active_post', 'demand_net_reactive_post'],
                               "demand_net_active_post", "demand_net_reactive_post"],
        'ZERO_BASE_FEATURES': [#"DHI", "GHI", "DNI"],
                               #"GHI"
                              ],
        'LOG1P_COLS':         ["n_lines", "tot_R_grid"],
        'TS_PERIODS':         [8760, 168, 24, 12
                               # 8760, 168, 24, 12, 8
                              ],
        'VMD_COLS':           [#'DHI', "DNI", "GHI", "T", "demand_net_active_pre"],
                               #'DHI', "DNI", "GHI", "T", "demand_net_active_pre", "PV_prod_expected"],
                               "demand_net_active_pre", "heat_water", "heat_space", "cop_avg", "PV_prod_expected", "T"],
                               #"demand_net_active_pre", "heat_water", "heat_space", "cop_avg", "mob_avail", "mobility", "PV_prod_expected"], #"PV_prod_100"],
        'VMD_K_MODES':         2,
        'VMD_APPROACH':        "write",  # None | read | write # read retrieves the VMD from cache, write makes it compute the VMD anew. "None" will have the VMD computed anew, but will not write it in the cache
    },
    # Model / architecture
    'input_mlp_layers': 2,
    'input_mlp_hidden': 256*2,
    'input_dropout': 0.1,
    'output_mlp_layers': 2,
    'output_mlp_hidden': 256*2,
    'output_dropout': 0.1,
    'd_model': 256*2,
    'num_heads': 8,
    'ff_dim': 256*8,
    'attn_dropout': 0,
    'ffn_dropout': 0,
    'num_transformer_layers': 2,
    'activation': 'gelu',
    # Aggregation settings
    'core_len': 120,
    'pad_hours': 24,
    'aggregation_mode': "conv",
    "conv_padding": 12,
    'agg_hours': 1,
    # Training
    'learning_rate': 1e-3,
    'weight_decay': 0.01,
    'optimizer': 'adamw',
    'scheduler': 'none',
    'scheduler_epochs': 40,
    'epochs': 40,
    'batch_size': 256,
    'grad_clip': 0,
    'loss_type': "alpha_peak", #'mae_maex',  # try: 'mae_maex', 'alpha_peak'
    'multi_target_lambda': 1,
    'patience': 3,
    # reporting and Others
    'seed': 42,
    "num_workers": 14,
    'compile_model': True,
    'bottom_rung_report': None,
    'full_metrics_every': 5,
}

transformer_trainer = TransformerTrainer(manual_cfg)
print('Train batches:', len(transformer_trainer.train_loader), ', Val batches:', len(transformer_trainer.val_loader))
print('Input dims ->', transformer_trainer.cfg.in_features, ', Output dims ->', transformer_trainer.cfg.out_features)

In [ ]:
import pandas as pd
def print_metrics(metrics):
    df=pd.DataFrame(metrics, index = [0])
    df.columns = [col[4:] if col.startswith('val_') else col for col in df.columns]
    def print_line(df, col_range):
        if not df.columns[col_range].empty: print(df[df.columns[col_range]].to_string(line_width=160, index=False))
    for cols in [slice(0,7), slice(7,12), slice(12,17), slice(17,22), slice(22,27), slice(27,32)]: print_line(df, cols)

for i in range(8):
    metrics = transformer_trainer.train_one_epoch()
    print_metrics(metrics)

/home/pedro/miniconda3/envs/training/lib/python3.12/site-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 14 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/pedro/miniconda3/envs/training/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/pedro/miniconda3/envs/training/lib/python3.12/site-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 14 worker processes in total. Our suggested max number of worker in current system is 12, which is s

 epoch  train_loss       loss  best_val_loss    lr
     1    2.502506 212.132812     212.132812 0.001


/home/pedro/miniconda3/envs/training/lib/python3.12/site-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 14 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/pedro/miniconda3/envs/training/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


 epoch  train_loss      loss  best_val_loss    lr
     2  166.544861 51.413883      51.413883 0.001


/home/pedro/miniconda3/envs/training/lib/python3.12/site-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 14 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/pedro/miniconda3/envs/training/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


 epoch  train_loss      loss  best_val_loss    lr
     3   44.914818 28.647636      28.647636 0.001


/home/pedro/miniconda3/envs/training/lib/python3.12/site-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 14 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/pedro/miniconda3/envs/training/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
transformer_trainer.evaluate_best_with_plots() ### test

In [ ]:
transformer_trainer.save("base_peak_EliasHPO.pt")
import os
os._exit(0)